# Import Data

In [1]:
# import findspark
import findspark
# initialize findspark with spark directory
findspark.init()
# import pyspark
import pyspark
# create spark contextp
sc = pyspark.SparkContext()
# create spark session
spark = pyspark.sql.SparkSession(sc)

In [2]:
spark

In [4]:
# import required packages
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# import pyspark functionalities
from pyspark.sql.types import NumericType
from pyspark.ml import Pipeline
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder, CountVectorizer, VectorSizeHint
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.linalg import VectorUDT, Vectors
from pyspark.ml.functions import vector_to_array

# turn of warnings
import warnings
warnings.filterwarnings("ignore")

In [5]:
basetable = spark.read.parquet("../Cleaned_data/biweekly_basetable_w_sentiment_5.parquet")
basetable.show(5)

+--------+---------------+--------------+------+--------------------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+-------------------+-------+---------+---------------------+------------------------+-----------------------+--------+-------------------+-------------------------+-------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+--------------------+------------------+------------------+---------+------------

In [6]:
basetable.count()

587

avg_Financing_sim_comp not ready yet so we'll drop it

basetable = basetable.drop("avg_Financing_sim_comp", "next_announcement_comp")
basetable.show(20)

# Modeling

## Time based split

In [10]:
# Determine the the cutoff biweekly index
max_index = basetable.agg(F.max("biweekly_index")).collect()[0][0]
cutoff_index = int(max_index * 0.67)

train_df = basetable.filter(F.col("biweekly_index") <= cutoff_index)
test_df  = basetable.filter(F.col("biweekly_index") > cutoff_index)

## Imputing Missing Values

In [11]:
# get all numeric columns
numeric_cols = [c for c, t in train_df.dtypes if isinstance(train_df.schema[c].dataType, NumericType)]

# count null values in each column
null_counts_train = train_df.select([F.count(F.when(F.col(c).isNull() | F.isnan(c), c)).alias(c) for c in numeric_cols])
null_counts_train.show()

null_counts_test = test_df.select([F.count(F.when(F.col(c).isNull() | F.isnan(c), c)).alias(c) for c in numeric_cols])
null_counts_test.show()

+--------------+-----------+------------+-----------+----------+---------+-------------+-----------------------+----+----------------+-----------------+------------------+------------------+------------+----------------+----------------+-----------------+------------------+--------------------+---------+------------+-------+-------+---------+---------------------+------------------------+-----------------------+---------+-------------------------+-------------+------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+-------------+-----------------+-------------+-------------+------------+------------+--------------+-----------------+--------------+--------------+--------------+------------------------+--------------------------+-----------------------+---------------

In [12]:
train_df = train_df.filter(F.col("prev_period_return").isNotNull())
train_df = train_df.fillna(0)

In [13]:
test_df = test_df.fillna(0)

## Select Features

In [14]:
train_df.printSchema()

root
 |-- industry: string (nullable = true)
 |-- biweekly_period: string (nullable = true)
 |-- biweekly_index: integer (nullable = true)
 |-- ticker: string (nullable = true)
 |-- longname: string (nullable = true)
 |-- benchmark: string (nullable = true)
 |-- period_start: date (nullable = true)
 |-- period_end: date (nullable = true)
 |-- period_open: double (nullable = false)
 |-- period_close: double (nullable = false)
 |-- period_high: double (nullable = false)
 |-- period_low: double (nullable = false)
 |-- avg_close: double (nullable = false)
 |-- period_return: double (nullable = false)
 |-- benchmark_period_return: double (nullable = false)
 |-- beta: double (nullable = false)
 |-- price_volatility: double (nullable = false)
 |-- return_volatility: double (nullable = false)
 |-- avg_intraday_range: double (nullable = false)
 |-- max_intraday_range: double (nullable = false)
 |-- total_volume: double (nullable = false)
 |-- avg_daily_volume: double (nullable = false)
 |-- max

In [15]:
label_col = "financing_next_period"
cat_cols = ["industry", "maturity", "locations"]
id_cols =  ["ticker", "biweekly_index","biweekly_period", "longname", "benchmark", "period_start", "period_end"]
exclude_cols = set(id_cols + [label_col])

feature_columns = [c for c in train_df.columns if c not in exclude_cols]
numeric_feature_cols = [col for col in feature_columns if col not in cat_cols]

sent_cols = [c for c in basetable.columns if c.endswith(("_ca", "_comp", "_ft", "_tijd"))]

Problem with min and max ollama, cast to double

In [16]:
train_df[sent_cols]

DataFrame[n_comments_ca: bigint, avg_vader_ca: double, std_vader_ca: double, avg_roberta_ca: double, median_roberta_ca: double, min_roberta_ca: double, max_roberta_ca: double, n_artices_comp: bigint, avg_num_words_title_comp: double, avg_num_words_content_comp: double, avg_polarity_title_comp: double, avg_polarity_content_comp: double, avg_subjectivity_title_comp: double, avg_subjectivity_content_comp: double, avg_ollama_comp: double, std_ollama_comp: double, min_ollama_comp: double, max_ollama_comp: double, avg_FPI_comp: double, std_FPI_comp: double, avg_finbert_vec_comp: vector, accum_FPI_comp: double, std_avg_FPI_last3_comp: double, prev_std_FPI_comp: double, prev_avg_FPI_comp: double, article_count_ft: bigint, total_words_ft: bigint, avg_words_ft: double, polarity_ft: double, subjectivity_ft: double, ollama_sentiment_ft: double, finbert_embedding_ft: vector, article_count_tijd: bigint, total_words_tijd: bigint, avg_words_tijd: double, polarity_tijd: double, subjectivity_tijd: doubl

___

cols_to_cast = ["min_ollama_comp", "max_ollama_comp"]

for c in cols_to_cast:
    basetable = basetable.withColumn(c, F.col(c).cast("double"))
    
basetable = basetable.drop("avg_finbert_array_comp")

basetable.write.mode("overwrite").parquet("../Cleaned_data/biweekly_basetable_w_sentiment.parquet")

___

## Scaling

First Scaling the ticker specific values (these can be significantly different from ticker to ticker, so these should be standardised per ticker)

In [17]:
scale_cols = [
    "period_open","period_close","period_high","period_low","avg_close",
    "price_volatility","avg_intraday_range","max_intraday_range",
    "total_volume","avg_daily_volume","max_daily_volume","volume_volatility",
    "avg_volume_up_days","avg_volume_down_days",
    "avg_gap_size","max_gap",
    "avg_prChg",
    "prev_price_volatility","prev_avg_daily_volume","prev_volume_volatility"
]
no_scale_cols = [c for c in numeric_feature_cols if c not in scale_cols and c not in sent_cols]

In [18]:
# Fit stats op TRAIN
agg_exprs = []
for c in scale_cols:
    agg_exprs += [
        F.avg(F.col(c)).alias(f"{c}__mean"),
        F.stddev_samp(F.col(c)).alias(f"{c}__std"),
    ]

stats_df = train_df.groupBy("ticker").agg(*agg_exprs)


def apply_per_ticker_zscore(df, stats_df, scale_cols):
    df = df.join(stats_df, on="ticker", how="left")
    zcols = []

    for c in scale_cols:
        mu = F.col(f"{c}__mean")
        sd = F.col(f"{c}__std")
        z  = f"{c}__z"

        df = df.withColumn(
            z,
            F.when(sd.isNull() | (sd == 0), F.lit(0.0))
             .otherwise((F.col(c) - mu) / sd)
        )
        zcols.append(z)

    # drop stats cols
    drop_cols = [f"{c}__mean" for c in scale_cols] + [f"{c}__std" for c in scale_cols]
    df = df.drop(*drop_cols)

    return df, zcols


train_s, zcols = apply_per_ticker_zscore(train_df, stats_df, scale_cols)
test_s,  _     = apply_per_ticker_zscore(test_df,  stats_df, scale_cols)

In [19]:
VA = VectorAssembler(inputCols=feature_columns, outputCol="features")

In [20]:
null_counts = train_s.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in train_s.columns
])

null_counts.show(truncate=False)

+------+--------+---------------+--------------+--------+---------+------------+----------+-----------+------------+-----------+----------+---------+-------------+-----------------------+----+----------------+-----------------+------------------+------------------+------------+----------------+----------------+-----------------+------------------+--------------------+---------+------------+-------+-------+---------+---------------------+------------------------+-----------------------+--------+---------+-------------------------+-------------+------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+-------------+-----------------+-------------+---------+-------------+------------+------------+--------------+-----------------+--------------+--------------+------------

Now Scaling of the Sentiment features

In [21]:
sent_VA = VectorAssembler(
    inputCols=sent_cols,
    outputCol="sent_vec",
    handleInvalid="keep"
)

In [22]:
sent_scaler = StandardScaler(
    inputCol="sent_vec",
    outputCol="sent_scaled",
    withStd=True,
    withMean=False   # keep sparse-friendly; sentiment is dense but this is safe
)

## Encode Categorical Features

In [23]:
# Indexers
industry_indexer = StringIndexer(inputCol="industry", outputCol="industry_idx", handleInvalid="keep")
maturity_indexer = StringIndexer(inputCol="maturity", outputCol="maturity_idx", handleInvalid="keep")

# One-hot encoder
ENC = OneHotEncoder(
    inputCols=["industry_idx", "maturity_idx"],
    outputCols=["industry_ohe", "maturity_ohe"],
    handleInvalid="keep"
)

In [24]:
CV = CountVectorizer(
    inputCol="locations",
    outputCol="locations_vec",
    binary=True,     # True = presence/absence; set False if counts matter
    minDF=2          # ignore super-rare locations (tune)
)

## Vector Assembling

In [25]:
VA_final = VectorAssembler(
    inputCols=["sent_scaled"] + [f"{c}_ohe" for c in ["industry", "maturity"]] + ["locations_vec"] + zcols + no_scale_cols,
    #  ^^^ adjust these to whatever you currently assemble
    outputCol="features"
)

In [26]:
VA_RF = VectorAssembler(
    inputCols=["sent_scaled"] + [f"{c}_ohe" for c in ["industry", "maturity"]] + ["locations_vec"] + numeric_feature_cols,
    #  ^^^ adjust these to whatever you currently assemble
    outputCol="features"
)

Weights

In [27]:
counts = (
    train_df
    .groupBy(label_col)
    .count()
    .withColumnRenamed("count", "n")
)

total = counts.agg(F.sum("n")).collect()[0][0]

weights = (
    counts
    .withColumn("weight", total / (2 * F.col("n")))
)

weights.show()
def apply_weights(df):
    return df.join(weights, on="financing_next_period", how="left")

train_w = apply_weights(train_s)
test_w  = apply_weights(test_s)

+---------------------+---+------------------+
|financing_next_period|  n|            weight|
+---------------------+---+------------------+
|                    1| 16|           9.96875|
|                    0|303|0.5264026402640264|
+---------------------+---+------------------+



# 3. Models

# 3.1 LR

___

problem with vector size of avg_finbert_vec_comp, problem is the null values.
Add flag variable and impute with zero vectors

basetable.show(5)

In [28]:
EMB_DIM = 768

@F.udf(VectorUDT())
def zero_vector():
    return Vectors.dense([0.0] * EMB_DIM)

basetable = basetable.withColumn(
    "finbert_missing",
    F.col("avg_finbert_vec_comp").isNull().cast("double")
)

basetable = basetable.withColumn(
    "avg_finbert_vec_comp",
    F.coalesce(F.col("avg_finbert_vec_comp"), zero_vector())
)

basetable.show()

basetable.count()

basetable.write.mode("overwrite").parquet("../Cleaned_data/biweekly_basetable_w_sentiment_5.parquet")

___

In [67]:
basetable.printSchema()

root
 |-- industry: string (nullable = true)
 |-- biweekly_period: string (nullable = true)
 |-- biweekly_index: integer (nullable = true)
 |-- ticker: string (nullable = true)
 |-- longname: string (nullable = true)
 |-- benchmark: string (nullable = true)
 |-- period_start: date (nullable = true)
 |-- period_end: date (nullable = true)
 |-- period_open: double (nullable = true)
 |-- period_close: double (nullable = true)
 |-- period_high: double (nullable = true)
 |-- period_low: double (nullable = true)
 |-- avg_close: double (nullable = true)
 |-- period_return: double (nullable = true)
 |-- benchmark_period_return: double (nullable = true)
 |-- beta: double (nullable = true)
 |-- price_volatility: double (nullable = true)
 |-- return_volatility: double (nullable = true)
 |-- avg_intraday_range: double (nullable = true)
 |-- max_intraday_range: double (nullable = true)
 |-- total_volume: double (nullable = true)
 |-- avg_daily_volume: double (nullable = true)
 |-- max_daily_volume:

In [29]:
vector_cols = [
    "avg_finbert_vec_comp",
    "finbert_embedding_ft"
]


In [30]:
hints = [
    VectorSizeHint(
        inputCol=c,
        size=768,
        handleInvalid="error"   # or "skip"
    )
    for c in vector_cols
]

In [31]:
length_df = (
    basetable
    .filter(F.col("avg_finbert_vec_comp").isNotNull())
    .select(
        F.size(vector_to_array("avg_finbert_vec_comp")).alias("vec_len")
    )
    .groupBy("vec_len")
    .count()
    .orderBy("vec_len")
)

length_df.show(truncate=False)

+-------+-----+
|vec_len|count|
+-------+-----+
|768    |587  |
+-------+-----+



In [32]:
length_df = (
    basetable
    .filter(F.col("finbert_embedding_ft").isNotNull())
    .select(
        F.size(vector_to_array("finbert_embedding_ft")).alias("vec_len")
    )
    .groupBy("vec_len")
    .count()
    .orderBy("vec_len")
)

length_df.show(truncate=False)

+-------+-----+
|vec_len|count|
+-------+-----+
|768    |587  |
+-------+-----+



In [72]:
# define linear regression model
LR = LogisticRegression(featuresCol="features", labelCol=label_col, elasticNetParam=0.8, regParam=0.001)

# fit model on train
lr_pipeline = Pipeline(stages=[
    industry_indexer,
    maturity_indexer,
    ENC,
    CV,
    *hints,
    sent_VA,
    sent_scaler,
    VA_final,
    LR
])

# Model training
lr_model = lr_pipeline.fit(train_s)

In [73]:
lr_predictions = lr_model.transform(test_s)

In [74]:
lr_predictions.select("period_start", "period_end", "financing_next_period", "prediction").show()

+------------+----------+---------------------+----------+
|period_start|period_end|financing_next_period|prediction|
+------------+----------+---------------------+----------+
|  2025-03-24|2025-04-04|                    0|       0.0|
|  2025-04-07|2025-04-17|                    0|       0.0|
|  2025-04-21|2025-05-02|                    0|       0.0|
|  2025-05-05|2025-05-16|                    0|       0.0|
|  2025-05-20|2025-05-30|                    0|       0.0|
|  2025-06-02|2025-06-13|                    0|       0.0|
|  2025-06-16|2025-06-27|                    0|       0.0|
|  2025-06-30|2025-07-11|                    0|       0.0|
|  2025-07-14|2025-07-25|                    0|       0.0|
|  2025-07-28|2025-08-08|                    0|       0.0|
|  2025-08-11|2025-08-22|                    0|       0.0|
|  2025-08-25|2025-08-29|                    0|       0.0|
|  2025-03-24|2025-04-04|                    0|       0.0|
|  2025-04-07|2025-04-17|                    0|       0.

In [100]:
def calculate_classification_metrics(predictions, label_col, prediction_col):
    # calculate true positives, true negatives, false positives, false negatives
    tp = predictions.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 1)).count()
    tn = predictions.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 0)).count()
    fp = predictions.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 1)).count()
    fn = predictions.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 0)).count()

    # calculate accuracy
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    
    # calculate precision
    precision = tp / (tp + fp) if (tp + fp) != 0 else 0  
    
    # calculate recall
    recall = tp / (tp + fn) if (tp + fn) != 0 else 0.0  
    
    # calculate F1 measure
    f1_measure = 2 * (precision * recall) / (precision + recall) if (precision + recall) != 0 else 0.0  

    # print the metrics
    print(f"Accuracy: {accuracy}")
    print(f"Precision: {precision}")
    print(f"Recall: {recall}")
    print(f"F1 measure: {f1_measure}")
    
    return accuracy, precision, recall, f1_measure

In [76]:
lr_accuracy, lr_precision, lr_recall, lr_f1_measure = calculate_classification_metrics(lr_predictions, 'financing_next_period', 'prediction')

Accuracy: 0.824
Precision: 0.11428571428571428
Recall: 0.23529411764705882
F1 measure: 0.15384615384615385


In [77]:
# get AUC
evaluator = BinaryClassificationEvaluator(labelCol="financing_next_period", 
                                          rawPredictionCol="probability", 
                                          metricName="areaUnderROC")

lr_AUC = evaluator.evaluate(lr_predictions)
print(f"AUC: {lr_AUC:.4f}")

AUC: 0.7137


# 3.2 LR with tuning

In [80]:
def walk_forward_eval(
    df_train,
    pipeline,
    time_col="biweekly_index",
    label_col="financing_next_period",
    metric="areaUnderPR",
    init_train_end=23,     # last period in first training window
    val_len=6,             # number of future periods to validate on
    step=2,                # how much the training window moves forward
    embargo=0,             # gap between train and validation (for lag features)
    threshold=0.2          # used only for recall/precision reporting
):
    """
    Performs expanding-window (walk-forward) evaluation.

    Returns:
        List of dicts with performance per window
    """

    evaluator = BinaryClassificationEvaluator(
        labelCol=label_col,
        metricName=metric
    )

    results = []
    max_time = df_train.agg(F.max(time_col)).collect()[0][0]

    train_end = init_train_end

    while True:
        # Define validation window
        val_start = train_end + 1 + embargo
        val_end   = val_start + val_len - 1

        if val_end > max_time:
            break

        # Slice train / validation
        train_w = df_train.filter(F.col(time_col) <= train_end)
        val_w   = df_train.filter(
            (F.col(time_col) >= val_start) &
            (F.col(time_col) <= val_end)
        )

        if val_w.count() == 0:
            train_end += step
            continue

        # Fit model on past
        model = pipeline.fit(train_w)

        # Predict future
        pred = model.transform(val_w)

        # ---------- PR-AUC ----------
        pr_auc = evaluator.evaluate(pred)

        # ---------- Recall / Precision at threshold ----------
        pred = pred.withColumn(
            "p1",
            vector_to_array("probability")[1]
        )

        pred = pred.withColumn(
            "pred_label",
            (F.col("p1") >= F.lit(threshold)).cast("int")
        )

        tp = pred.filter((F.col(label_col) == 1) & (F.col("pred_label") == 1)).count()
        fp = pred.filter((F.col(label_col) == 0) & (F.col("pred_label") == 1)).count()
        fn = pred.filter((F.col(label_col) == 1) & (F.col("pred_label") == 0)).count()
        tn = pred.filter((F.col(label_col) == 0) & (F.col("pred_label") == 0)).count()

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall    = tp / (tp + fn) if (tp + fn) else 0.0

        results.append({
            "train_end": int(train_end),
            "val_start": int(val_start),
            "val_end": int(val_end),
            "areaUnderPR": float(pr_auc),
            "precision@thr": float(precision),
            "recall@thr": float(recall),
            "tp": int(tp),
            "fp": int(fp),
            "fn": int(fn),
            "tn": int(tn)
        })

        # Move window forward
        train_end += step

    return results


In [78]:
from pyspark.ml.tuning import ParamGridBuilder


In [ ]:
def build_lr_pipeline(params):
    lr = LogisticRegression(
        featuresCol="features",
        labelCol="financing_next_period",
        probabilityCol="probability",
        rawPredictionCol="rawPrediction",
        predictionCol="prediction",
        **params,
        weightCol="weight"
    )
    return Pipeline(stages=[
        industry_indexer,
        maturity_indexer,
        ENC,
        CV,
        *hints,        # only if you still have vector columns needing VectorSizeHint
        sent_VA,
        sent_scaler,
        VA_final,
        lr
    ])

In [82]:
reg_grid = [float(x) for x in np.logspace(-6, -1, 6)]  # 1e-6 ... 1e-1
enet_grid = [0.0, 0.25, 0.5, 0.75, 1.0]
maxIter = [100, 200]

param_grid = []
for reg in reg_grid:
    for enet in enet_grid:
        for iter in maxIter:  
            param_grid.append({
                "regParam": reg,
                "elasticNetParam": enet,
                "maxIter": iter,
                "tol": 1e-6,
                "standardization": True,
            })

In [93]:
best = None
all_results = []

for params in param_grid:
    pipe = build_lr_pipeline(params)

    wf_rows = walk_forward_eval(   # your fixed-range version (0..47)
        df_train=train_w,
        pipeline=pipe,
        time_col="biweekly_index",
        label_col="financing_next_period",
        metric="areaUnderPR",
        init_train_end=23,   # train 0..23
        val_len=6,
        step=2,
        embargo=0,
        threshold=0.2        # only used if your evaluator also logs threshold metrics
    )

    avg_pr = sum(r["areaUnderPR"] for r in wf_rows) / len(wf_rows)
    avg_recall = sum(r["recall@thr"] for r in wf_rows) / len(wf_rows)

    all_results.append((params, avg_pr, avg_recall))

    # OPTION A: prioritize PR-AUC, break ties with recall
    if (best is None) or (avg_pr > best[1]) or (
        avg_pr == best[1] and avg_recall > best[2]
    ):
        best = (params, avg_pr, avg_recall)

print("BEST LR PARAMS:", best[0])
print("Avg walk-forward PR-AUC:", best[1])
print("Avg walk-forward Recall@0.2:", best[2])

BEST LR PARAMS: {'regParam': 0.01, 'elasticNetParam': 1.0, 'maxIter': 100, 'tol': 1e-06, 'standardization': True}
Avg walk-forward PR-AUC: 0.3263994215572408
Avg walk-forward Recall@0.2: 0.4


In [94]:
all_results

[({'regParam': 1e-06,
   'elasticNetParam': 0.0,
   'maxIter': 100,
   'tol': 1e-06,
   'standardization': True},
  0.13091752363133943,
  0.6),
 ({'regParam': 1e-06,
   'elasticNetParam': 0.0,
   'maxIter': 200,
   'tol': 1e-06,
   'standardization': True},
  0.1494361972576747,
  0.4),
 ({'regParam': 1e-06,
   'elasticNetParam': 0.25,
   'maxIter': 100,
   'tol': 1e-06,
   'standardization': True},
  0.17328909229595726,
  0.4),
 ({'regParam': 1e-06,
   'elasticNetParam': 0.25,
   'maxIter': 200,
   'tol': 1e-06,
   'standardization': True},
  0.1726901027334434,
  0.4),
 ({'regParam': 1e-06,
   'elasticNetParam': 0.5,
   'maxIter': 100,
   'tol': 1e-06,
   'standardization': True},
  0.13778060938508702,
  0.2),
 ({'regParam': 1e-06,
   'elasticNetParam': 0.5,
   'maxIter': 200,
   'tol': 1e-06,
   'standardization': True},
  0.14605426003313327,
  0.2),
 ({'regParam': 1e-06,
   'elasticNetParam': 0.75,
   'maxIter': 100,
   'tol': 1e-06,
   'standardization': True},
  0.13493748127

test

In [85]:
# define linear regression model
LR_best = LogisticRegression(
    featuresCol="features", 
    labelCol=label_col, 
    elasticNetParam=1.0, 
    regParam=0.01, 
    maxIter=100,
    tol=1e-06,
    standardization=True,
    weightCol="weight"
)

# fit model on train
lr_pipeline = Pipeline(stages=[
    industry_indexer,
    maturity_indexer,
    ENC,
    CV,
    *hints,
    sent_VA,
    sent_scaler,
    VA_final,
    LR_best
])

# Model training
lr_model = lr_pipeline.fit(train_w)

In [86]:
lr_predictions = lr_model.transform(test_w)

In [101]:
lr_accuracy, lr_precision, lr_recall, lr_f1_measure = calculate_classification_metrics(lr_predictions, 'financing_next_period', 'prediction')

Accuracy: 0.692
Precision: 0.0945945945945946
Recall: 0.4117647058823529
F1 measure: 0.15384615384615383


In [ ]:
#BEST LR PARAMS: {'regParam': 0.01, 'elasticNetParam': 1.0, 'maxIter': 100, 'tol': 1e-06, 'standardization': True}

In [98]:
lr_predictions.show()

+---------------------+------+--------------------+---------------+--------------+--------------------+---------+------------+----------+-----------+------------+-----------+----------+-------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+------------------------+-----------------------+--------+--------------------+-------------------------+-------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+--------------------+-------------------+-------------------+-----

In [102]:
lr_predictions.write.mode("overwrite").parquet("../Models/Results/lr_predictions_1")

# LR with recall tuning

In [95]:
def recall_at_k(pred_df, label_col, period_col, k=0.10):
    """
    Recall@K where K is a fraction (e.g. 0.10 means top 10%) per period.
    Ranking is done within each period_col.
    """
    df = pred_df.withColumn("p1", vector_to_array("probability")[1])

    w_rank = Window.partitionBy(period_col).orderBy(F.col("p1").desc())
    w_cnt  = Window.partitionBy(period_col)

    ranked = (df
        .withColumn("rank", F.row_number().over(w_rank))
        .withColumn("n", F.count("*").over(w_cnt))
        .withColumn("is_top_k", (F.col("rank") <= F.col("n") * F.lit(k)).cast("int"))
    )

    tp = ranked.filter((F.col("is_top_k") == 1) & (F.col(label_col) == 1)).count()
    total_pos = ranked.filter(F.col(label_col) == 1).count()

    return tp / total_pos if total_pos else 0.0

In [96]:
def walk_forward_eval_pr_and_recallk(
    df_train,
    pipeline,
    time_col="biweekly_index",
    label_col="financing_next_period",
    init_train_end=23,   # first train window is 0..23
    val_len=6,           # validate on next 6 periods
    step=2,              # slide forward
    embargo=0,           # gap between train and val (set to max lag if needed)
    k=0.10               # Recall@K (top 10% per period)
):
    evaluator = BinaryClassificationEvaluator(labelCol=label_col, metricName="areaUnderPR")

    max_time = df_train.agg(F.max(time_col)).collect()[0][0]

    rows = []
    train_end = init_train_end

    while True:
        val_start = train_end + 1 + embargo
        val_end   = val_start + val_len - 1
        if val_end > max_time:
            break

        train_w = df_train.filter(F.col(time_col) <= train_end)
        val_w   = df_train.filter((F.col(time_col) >= val_start) & (F.col(time_col) <= val_end))

        # fit on past, predict future
        model = pipeline.fit(train_w)
        pred  = model.transform(val_w)

        pr_auc = evaluator.evaluate(pred)
        rec_k  = recall_at_k(pred, label_col=label_col, period_col=time_col, k=k)

        rows.append({
            "train_end": int(train_end),
            "val_start": int(val_start),
            "val_end": int(val_end),
            "pr_auc": float(pr_auc),
            f"recall@{int(k*100)}%": float(rec_k)
        })

        train_end += step

    return rows


In [97]:
K = 0.10  # 10% alerts per biweek

best = None
all_results = []

for params in param_grid:
    pipe = build_lr_pipeline(params)  # or build_gbt_pipeline(params)

    wf = walk_forward_eval_pr_and_recallk(
        df_train=train_s,
        pipeline=pipe,
        time_col="biweekly_index",
        label_col="financing_next_period",
        init_train_end=23,
        val_len=6,
        step=2,
        embargo=0,    # set to max lag if you use lag features
        k=K
    )

    avg_pr = sum(r["pr_auc"] for r in wf) / len(wf)
    avg_rk = sum(r[f"recall@{int(K*100)}%"] for r in wf) / len(wf)

    all_results.append((params, avg_pr, avg_rk))

    if (best is None) or (avg_pr > best[1]) or (avg_pr == best[1] and avg_rk > best[2]):
        best = (params, avg_pr, avg_rk)

print("BEST PARAMS:", best[0])
print("Avg walk-forward PR-AUC:", best[1])
print(f"Avg walk-forward Recall@{int(K*100)}%:", best[2])


BEST PARAMS: {'regParam': 0.01, 'elasticNetParam': 1.0, 'maxIter': 100, 'tol': 1e-06, 'standardization': True}
Avg walk-forward PR-AUC: 0.3263994215572408
Avg walk-forward Recall@10%: 0.4


# 3.3 RF

In [80]:
# define random forest model
RF = RandomForestClassifier(featuresCol="features", labelCol="financing_next_period", numTrees=100)

# fit model on train
rf_pipeline = Pipeline(stages=[
    industry_indexer,
    maturity_indexer,
    ENC,
    CV,
    *hints, # locations array -> locations_vec (CountVectorizer)
    sent_VA,
    sent_scaler,
    VA_RF,
    RF
])

# model training
rf_model = rf_pipeline.fit(train_df)

In [81]:
rf_predictions = rf_model.transform(test_df)

In [82]:
# show predictions
rf_predictions.select("period_start", "period_end", "financing_next_period", "prediction").show()

+------------+----------+---------------------+----------+
|period_start|period_end|financing_next_period|prediction|
+------------+----------+---------------------+----------+
|  2025-03-24|2025-04-04|                    0|       0.0|
|  2025-04-07|2025-04-17|                    0|       0.0|
|  2025-04-21|2025-05-02|                    0|       0.0|
|  2025-05-05|2025-05-16|                    0|       0.0|
|  2025-05-20|2025-05-30|                    0|       0.0|
|  2025-06-02|2025-06-13|                    0|       0.0|
|  2025-06-16|2025-06-27|                    0|       0.0|
|  2025-06-30|2025-07-11|                    0|       0.0|
|  2025-07-14|2025-07-25|                    0|       0.0|
|  2025-07-28|2025-08-08|                    0|       0.0|
|  2025-08-11|2025-08-22|                    0|       0.0|
|  2025-08-25|2025-08-29|                    0|       0.0|
|  2025-03-24|2025-04-04|                    0|       0.0|
|  2025-04-07|2025-04-17|                    0|       0.

In [83]:
rf_accuracy, rf_precision, rf_recall, rf_f1_measure = calculate_classification_metrics(rf_predictions, 'financing_next_period', 'prediction')

Accuracy: 0.932
Precision: 0
Recall: 0.0
F1 measure: 0.0


In [84]:
# get AUC
rf_AUC = evaluator.evaluate(rf_predictions)
print(f"AUC: {rf_AUC:.4f}")

AUC: 0.6039


# GBT

In [86]:
# define boosting model
GBT = GBTClassifier(
    featuresCol="features", 
    labelCol="financing_next_period",
    maxDepth=3,                 # SHALLOW
    maxIter=40,                 # few trees
    stepSize=0.05,              # conservative learning
    subsamplingRate=0.7,        # reduce variance
    minInstancesPerNode=20,     # avoid tiny leafs
    seed=42
)

# fit model on train
gbt_pipeline = Pipeline(stages=[
    industry_indexer,
    maturity_indexer,
    ENC,
    CV,
    *hints, # locations array -> locations_vec (CountVectorizer)
    sent_VA,
    sent_scaler,
    VA_final,
    GBT
])

# model training
gbt_model = gbt_pipeline.fit(train_s)

In [87]:
gbt_predictions = gbt_model.transform(test_s)

In [88]:
gbt_predictions.select("period_start", "period_end", "financing_next_period", "prediction").show()

+------------+----------+---------------------+----------+
|period_start|period_end|financing_next_period|prediction|
+------------+----------+---------------------+----------+
|  2025-03-24|2025-04-04|                    0|       0.0|
|  2025-04-07|2025-04-17|                    0|       0.0|
|  2025-04-21|2025-05-02|                    0|       0.0|
|  2025-05-05|2025-05-16|                    0|       0.0|
|  2025-05-20|2025-05-30|                    0|       0.0|
|  2025-06-02|2025-06-13|                    0|       0.0|
|  2025-06-16|2025-06-27|                    0|       0.0|
|  2025-06-30|2025-07-11|                    0|       0.0|
|  2025-07-14|2025-07-25|                    0|       0.0|
|  2025-07-28|2025-08-08|                    0|       0.0|
|  2025-08-11|2025-08-22|                    0|       0.0|
|  2025-08-25|2025-08-29|                    0|       0.0|
|  2025-03-24|2025-04-04|                    0|       0.0|
|  2025-04-07|2025-04-17|                    0|       0.

In [89]:
gbt_accuracy, gbt_precision, gbt_recall, gbt_f1_measure = calculate_classification_metrics(gbt_predictions, 'financing_next_period', 'prediction')

Accuracy: 0.932
Precision: 0
Recall: 0.0
F1 measure: 0.0


In [90]:
# get AUC
gbt_AUC = evaluator.evaluate(gbt_predictions)
print(f"AUC: {gbt_AUC:.4f}")

AUC: 0.6074


# GBT with tuning

In [36]:
# -----------
# DEFINE FUNCTION: a function that calculates all the evaluation metrics
# -----------
def calculate_classification_metrics(predictions, label_col, prediction_col):
    # calculate true positives, true negatives, false positives, false negatives
    tp = predictions.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 1)).count()
    tn = predictions.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 0)).count()
    fp = predictions.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 1)).count()
    fn = predictions.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 0)).count()

    # calculate accuracy
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    
    # calculate precision
    precision = tp / (tp + fp) if (tp + fp) != 0 else 0  
    
    # calculate recall
    recall = tp / (tp + fn) if (tp + fn) != 0 else 0.0  
    
    # calculate F1 measure
    f1_measure = 2 * (precision * recall) / (precision + recall) if (precision + recall) != 0 else 0.0  

    # print the metrics
    #print(f"Accuracy: {accuracy}")
    #print(f"Precision: {precision}")
    #print(f"Recall: {recall}")
    #print(f"F1 measure: {f1_measure}")
    
    return accuracy, precision, recall, f1_measure

# -----------
# DEFINE FUNCTION: a function that makes custom predictions based on a given threshold
# -----------    
def custom_predictions(threshold, df, score_col):
    
    return df.withColumn(
        "custom_prediction",
        F.when(F.col(score_col) >= F.lit(threshold), 1).otherwise(0)
    )
    

def optimize_threshold(
    thresholds, 
    pred_df,
    score_col,
    cost_accuracy, cost_precision, cost_recall, cost_f1,
    label_col="financing_next_period"
):
    # create evaluation frame
    total_cost = []
    accuracy_list = []
    precision_list = []
    recall_list = []
    f1_list = []

    cut_off_values = []
    for threshold in thresholds:
        evaluation_frame = custom_predictions(threshold, pred_df, score_col)

        accuracy, precision, recall, f1_measure = calculate_classification_metrics(
            evaluation_frame, label_col, "custom_prediction"
        )

        total_cost_i = (accuracy * cost_accuracy +
                        precision * cost_precision +
                        recall * cost_recall +
                        f1_measure * cost_f1)

        total_cost.append(total_cost_i)
        accuracy_list.append(accuracy)
        precision_list.append(precision)
        recall_list.append(recall)
        f1_list.append(f1_measure)
        cut_off_values.append(threshold)

    max_profit = max(total_cost)
    max_index = total_cost.index(max_profit)
    best_cut_off = cut_off_values[max_index]

    print(f"The best threshold is {best_cut_off}, with a score of {max_profit}")
    print(f"Accuracy:  {accuracy_list[max_index]}")
    print(f"Precision: {precision_list[max_index]}")
    print(f"Recall:    {recall_list[max_index]}")
    print(f"F1 measure:{f1_list[max_index]}")

    return best_cut_off

In [37]:
thresholds = [i/100 for i in range(1, 60)]  # 0.01..0.59 (adjust)

In [41]:
# define boosting model
GBT = GBTClassifier(
    featuresCol="features", 
    labelCol="financing_next_period",
    seed=42
)

# fit model on train
gbt_pipeline = Pipeline(stages=[
    industry_indexer,
    maturity_indexer,
    ENC,
    CV,
    *hints, # locations array -> locations_vec (CountVectorizer)
    sent_VA,
    sent_scaler,
    VA_final,
    GBT
])

# model training
gbt_model = gbt_pipeline.fit(train_s)

gbt_predictions = gbt_model.transform(test_s)

eval_df = gbt_predictions.select("ticker", "biweekly_period", "financing_next_period", "prediction", "probability").withColumn("p_pos", vector_to_array("probability")[1])
eval_df.show()

+------+---------------+---------------------+----------+--------------------+--------------------+
|ticker|biweekly_period|financing_next_period|prediction|         probability|               p_pos|
+------+---------------+---------------------+----------+--------------------+--------------------+
|ARG.TO|            P31|                    0|       0.0|[0.95152796499628...| 0.04847203500371777|
|ARG.TO|            P32|                    0|       0.0|[0.95585894235994...| 0.04414105764005216|
|ARG.TO|            P33|                    0|       0.0|[0.96593994352448...| 0.03406005647551302|
|ARG.TO|            P34|                    0|       0.0|[0.95585894235994...| 0.04414105764005216|
|ARG.TO|            P35|                    0|       1.0|[0.27883555941116...|  0.7211644405888314|
|ARG.TO|            P36|                    0|       0.0|[0.95700078773801...| 0.04299921226198944|
|ARG.TO|            P37|                    0|       0.0|[0.95585894235994...| 0.04414105764005216|


In [46]:
best_t = optimize_threshold(
    thresholds, eval_df,
    score_col="p_pos",
    cost_accuracy=1, cost_precision=1, cost_recall=1.01, cost_f1=1
)

The best threshold is 0.07, with a score of 1.3151197439576205
Accuracy:  0.72
Precision: 0.09230769230769231
Recall:    0.35294117647058826
F1 measure:0.14634146341463414


Recall cost = 1
The best threshold is 0.07, with a score of 1.3115903321929145
Accuracy:  0.72
Precision: 0.09230769230769231
Recall:    0.35294117647058826
F1 measure:0.14634146341463414

In [35]:
evaluator = BinaryClassificationEvaluator(labelCol="financing_next_period", 
                                          rawPredictionCol="probability", 
                                          metricName="areaUnderROC")
gbt_AUC = evaluator.evaluate(gbt_predictions)
print(f"AUC: {gbt_AUC:.4f}")

AUC: 0.5871


---

In [47]:
def choose_threshold_max_recall_with_precision_floor(
    thresholds, pred_df, score_col,
    min_precision=0.15,
    label_col="financing_next_period"
):
    best = None  # (threshold, recall, precision, f1, acc)
    for t in thresholds:
        ev = custom_predictions(t, pred_df, score_col)
        acc, prec, rec, f1 = calculate_classification_metrics(ev, label_col, "custom_prediction")

        if prec >= min_precision:
            cand = (t, rec, prec, f1, acc)
            if (best is None) or (cand[1] > best[1]) or (cand[1] == best[1] and cand[2] > best[2]):
                best = cand

    if best is None:
        print(f"No threshold meets precision >= {min_precision}. Try lowering min_precision or improving model.")
        return None

    t, rec, prec, f1, acc = best
    print(f"Chosen threshold: {t}")
    print(f"Precision: {prec} | Recall: {rec} | F1: {f1} | Accuracy: {acc}")
    return t

In [49]:
thresholds = [i/1000 for i in range(1, 300)]  # 0.001..0.299 (finer grid helps)
best_t = choose_threshold_max_recall_with_precision_floor(
    thresholds, eval_df, score_col="p_pos", min_precision=0.10
)


Chosen threshold: 0.079
Precision: 0.1 | Recall: 0.35294117647058826 | F1: 0.15584415584415584 | Accuracy: 0.74


___

In [44]:
def build_gbt_pipeline(params):
    gbt = GBTClassifier(
        featuresCol="features",
        labelCol="financing_next_period",
        **params,
        weightCol="weight"
    )
    return Pipeline(stages=[
        industry_indexer,
        maturity_indexer,
        ENC,
        CV,
        *hints,
        sent_VA,
        sent_scaler,
        VA_final,
        gbt
    ])

In [45]:
param_grid = [
    dict(maxDepth=2, maxIter=40, stepSize=0.05, subsamplingRate=0.7, minInstancesPerNode=20),
    dict(maxDepth=3, maxIter=40, stepSize=0.05, subsamplingRate=0.7, minInstancesPerNode=20),
    dict(maxDepth=3, maxIter=80, stepSize=0.05, subsamplingRate=0.7, minInstancesPerNode=20),
    dict(maxDepth=3, maxIter=40, stepSize=0.03, subsamplingRate=0.7, minInstancesPerNode=20),
]

In [51]:
best = {'maxDepth': 3, 'maxIter': 80, 'stepSize': 0.05, 'subsamplingRate': 0.8, 'minInstancesPerNode': 20, "seed":42}

param_grid2 = [
    # stepSize/maxIter neighborhood
    {**best, "stepSize": 0.03, "maxIter": 160},
    {**best, "stepSize": 0.03, "maxIter": 120},
    {**best, "stepSize": 0.05, "maxIter": 120},
    {**best, "stepSize": 0.07, "maxIter": 80},
    {**best, "stepSize": 0.10, "maxIter": 60},

    # subsampling neighborhood
    {**best, "subsamplingRate": 0.75},
    {**best, "subsamplingRate": 0.80},
    {**best, "subsamplingRate": 0.85},
    {**best, "subsamplingRate": 0.90},    

    # complexity neighborhood
    {**best, "maxDepth": 2},
    {**best, "maxDepth": 4, "minInstancesPerNode": 40},
    {**best, "minInstancesPerNode": 10},
    {**best, "minInstancesPerNode": 20},
    {**best, "minInstancesPerNode": 40},
]

In [40]:
def walk_forward_eval(
    df_train,
    pipeline,                      # your Pipeline (already built)
    time_col="biweekly_index",
    label_col="financing_next_period",
    metric="areaUnderPR",
    init_train_periods=30,         # initial number of biweeks to train on
    val_periods=6,                 # validation window size (biweeks)
    step=2,                        # slide forward by this many biweeks
    embargo=0,                     # gap between train end and val start (useful with lags)
    threshold=0.2                  # for precision/recall at a fixed operating point
):
    # sorted unique time points present in df_train
    times = [r[0] for r in df_train.select(time_col).distinct().orderBy(time_col).collect()]
    n = len(times)

    evaluator = BinaryClassificationEvaluator(labelCol=label_col, metricName=metric)

    rows = []
    # train uses [0 .. train_end], val uses [val_start .. val_end]
    train_end_pos = init_train_periods - 1

    while True:
        val_start_pos = train_end_pos + 1 + embargo
        val_end_pos   = val_start_pos + val_periods - 1
        if val_end_pos >= n:
            break

        train_end = times[train_end_pos]
        val_start = times[val_start_pos]
        val_end   = times[val_end_pos]

        train_w = df_train.filter(F.col(time_col) <= F.lit(train_end))
        val_w   = df_train.filter((F.col(time_col) >= F.lit(val_start)) & (F.col(time_col) <= F.lit(val_end)))

        model = pipeline.fit(train_w)
        pred  = model.transform(val_w)

        score = evaluator.evaluate(pred)

        # precision/recall at chosen threshold using prob(class=1)
        pred_bin = pred.withColumn(
            "pred_label",
            F.when(vector_to_array("probability")[1] >= F.lit(threshold), F.lit(1)).otherwise(F.lit(0))
        )

        # confusion matrix counts
        cm = (pred_bin.groupBy(label_col, "pred_label").count().collect())
        counts = {(r[label_col], r["pred_label"]): r["count"] for r in cm}
        tp = counts.get((1, 1), 0)
        fp = counts.get((0, 1), 0)
        fn = counts.get((1, 0), 0)
        tn = counts.get((0, 0), 0)

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall    = tp / (tp + fn) if (tp + fn) else 0.0

        rows.append({
            "train_end": int(train_end),
            "val_start": int(val_start),
            "val_end": int(val_end),
            metric: float(score),
            "precision@thr": float(precision),
            "recall@thr": float(recall),
            "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn)
        })

        train_end_pos += step

    return rows

In [48]:
best = None
all_results = []

for i, params in enumerate(param_grid2, 1):
    pipe = build_gbt_pipeline(params)

    wf_rows = walk_forward_eval(   # or your walk_forward_eval if it runs
        df_train=train_w,
        pipeline=pipe,
        time_col="biweekly_index",
        label_col="financing_next_period",
        metric="areaUnderPR",
        init_train_periods=23,   # 0..23 train
        val_periods=6,
        step=2,
        embargo=0,           # set to max lag if you use lag features
        threshold=0.2
    )

    avg_pr = sum(r["areaUnderPR"] for r in wf_rows) / len(wf_rows)
    avg_recall = sum(r["recall@thr"] for r in wf_rows) / len(wf_rows)
    
    print(f"{i:02d} params={params}  avgPR={avg_pr:.12f}  avgRecall@0.2={avg_recall:.12f}")

    all_results.append((params, avg_pr, avg_recall, len(wf_rows)))

    if (best is None) or (avg_pr > best[1]):
        best = (params, avg_pr, avg_recall)

print("BEST PARAMS:", best[0])
print("Avg walk-forward PR-AUC:", best[1])
print("Avg walk-forward Recall@0.2:", best[2])


01 params={'maxDepth': 3, 'maxIter': 160, 'stepSize': 0.03, 'subsamplingRate': 0.8, 'minInstancesPerNode': 20}  avgPR=0.339958868305  avgRecall@0.2=0.400000000000
02 params={'maxDepth': 3, 'maxIter': 120, 'stepSize': 0.03, 'subsamplingRate': 0.8, 'minInstancesPerNode': 20}  avgPR=0.339859816637  avgRecall@0.2=0.400000000000
03 params={'maxDepth': 3, 'maxIter': 120, 'stepSize': 0.05, 'subsamplingRate': 0.8, 'minInstancesPerNode': 20}  avgPR=0.425465100524  avgRecall@0.2=0.400000000000
04 params={'maxDepth': 3, 'maxIter': 80, 'stepSize': 0.07, 'subsamplingRate': 0.8, 'minInstancesPerNode': 20}  avgPR=0.279804507267  avgRecall@0.2=0.400000000000
05 params={'maxDepth': 3, 'maxIter': 60, 'stepSize': 0.1, 'subsamplingRate': 0.8, 'minInstancesPerNode': 20}  avgPR=0.308615955207  avgRecall@0.2=0.400000000000
06 params={'maxDepth': 3, 'maxIter': 80, 'stepSize': 0.05, 'subsamplingRate': 0.75, 'minInstancesPerNode': 20}  avgPR=0.240881121132  avgRecall@0.2=0.400000000000
07 params={'maxDepth': 3,

Best model

In [52]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import GBTClassifier

final_gbt = GBTClassifier(
    featuresCol="features",
    labelCol="financing_next_period",
    **best,
    weightCol="weight"
)

final_pipeline = Pipeline(stages=[
    industry_indexer,
    maturity_indexer,
    ENC,
    CV,
    *hints,          # VectorSizeHint stages
    sent_VA,
    sent_scaler,
    VA_final,        # VectorAssembler that creates "features"
    final_gbt
])

In [53]:
final_gbt_model = final_pipeline.fit(train_w)

In [54]:
final_gbt_predictions = final_gbt_model.transform(test_w)

In [55]:
def calculate_classification_metrics(predictions, label_col, threshold=0.2):
    preds = predictions.withColumn(
        "pred_label",
        (vector_to_array("probability")[1] >= F.lit(threshold)).cast("int")
    )

    tp = preds.filter((F.col(label_col) == 1) & (F.col("pred_label") == 1)).count()
    tn = preds.filter((F.col(label_col) == 0) & (F.col("pred_label") == 0)).count()
    fp = preds.filter((F.col(label_col) == 0) & (F.col("pred_label") == 1)).count()
    fn = preds.filter((F.col(label_col) == 1) & (F.col("pred_label") == 0)).count()

    accuracy  = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall    = tp / (tp + fn) if (tp + fn) else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    print(f"Threshold: {threshold}")
    print(f"TP={tp}, FP={fp}, FN={fn}, TN={tn}")
    print(f"Accuracy:  {accuracy:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")
    print(f"F1:        {f1:.3f}")

    return accuracy, precision, recall, f1


In [56]:
gbt_accuracy, gbt_precision, gbt_recall, gbt_f1 = calculate_classification_metrics(
    final_gbt_predictions,
    label_col="financing_next_period",
    threshold=0.2
)


Threshold: 0.2
TP=7, FP=58, FN=10, TN=175
Accuracy:  0.728
Precision: 0.108
Recall:    0.412
F1:        0.171


In [57]:
final_gbt_predictions.write.mode("overwrite").parquet("../Models/Results/gbt_predictions_1.parquet")

In [68]:
final_gbt_predictions = final_gbt_predictions.withColumn(
    "p1",
    vector_to_array("probability")[1]
)

In [70]:
from pyspark.sql.window import Window

w = Window.partitionBy("biweekly_index").orderBy(F.col("p1").desc())

final_gbt_predictions = final_gbt_predictions.withColumn("rank", F.row_number().over(w))


In [75]:
K = 0.50  # 10%

final_gbt_predictions = final_gbt_predictions.withColumn(
    "n_in_period",
    F.count("*").over(Window.partitionBy("biweekly_index"))
)

final_gbt_predictions = final_gbt_predictions.withColumn(
    "is_top_k",
    F.when(F.col("rank") <= F.col("n_in_period") * F.lit(K), 1).otherwise(0)
)

In [76]:
tp_k = final_gbt_predictions.filter(
    (F.col("is_top_k") == 1) & (F.col("financing_next_period") == 1)
).count()

total_pos = final_gbt_predictions.filter(F.col("financing_next_period") == 1).count()

recall_at_k = tp_k / total_pos if total_pos else 0.0

print(f"Recall@{int(K*100)}% = {recall_at_k:.3f}")

Recall@50% = 0.588
